In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))
if PROJECT_ROOT.name == "notebooks":
    sys.path.append(str(PROJECT_ROOT.parent))
    
    
from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble import ensemble
from src.model.game_model_collection import GamingModelCollection

# instantiate preprocessor and labeler
gaming_preprocessor = GamingTextPreprocessor()
gaming_labeler = GamingTextLabeler()

In [2]:
# config
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'processed_data').exists() and PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

print('CONFIG loaded. Text column:', tc)


CONFIG loaded. Text column: message


In [ ]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

def _safe_confusion_rates(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'FPR': fp / (fp + tn) if (fp + tn) > 0 else 0,
        'FNR': fn / (fn + tp) if (fn + tp) > 0 else 0,
        'TPR': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'TNR': tn / (tn + fp) if (tn + fp) > 0 else 0,
    }
    
def score_func(y_true, y_pred, uncertain_label=-1, min_coverage=0.80):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    covered = y_pred != uncertain_label
    coverage = covered.mean()

    if coverage < min_coverage:
        return -np.inf

    return f1_score(
        y_true[covered],
        y_pred[covered],
        average="macro"
    )

def precision(y_true, y_pred):
    rates = _safe_confusion_rates(y_true, y_pred)
    return rates['TPR'] / (rates['TPR'] + rates['FPR']) if (rates['TPR'] + rates['FPR']) > 0 else 0

def _prediction_metrics(y_true, y_pred):
    return {
        'CV Macro F1': f1_score(y_true, y_pred, average='macro'),
        'CV Weighted F1': f1_score(y_true, y_pred, average='weighted'),
        'Accuracy': accuracy_score(y_true, y_pred),
        "Precision": precision(y_true, y_pred),
        **_safe_confusion_rates(y_true, y_pred),
    }

# Fit the Gaming Text Preprocessor 

In [4]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

In [5]:
# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"

In [6]:
# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

In [7]:
# train 
gaming_preprocessor.fit(train, text_column=tc)

In [8]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

val = pd.concat([wot_val, dota_val], ignore_index=True)

# Label Cleaning

In [9]:

# convert dota labels 
train = gaming_labeler.convert_three_class(
    train, 
    label_column=lc, 
    data_source_column='data_source',
    output_column='label_3class'
)
val = gaming_labeler.convert_three_class(
    val, 
    label_column=lc, 
    data_source_column='data_source',
    output_column='label_3class'
)


In [10]:
train["label_3class"].value_counts()

label_3class
0    40929
1     7492
2     5286
Name: count, dtype: int64

# Model Loading

In [11]:
# load all models from disk
model_dir = CONFIG['model_dir'] / "multi"
model_paths = list(model_dir.glob("gaming_all_*.joblib"))
models = [joblib.load(path) for path in model_paths]

models

[{'wot': {'pipes': {'Logistic Regression': Pipeline(steps=[('tfidf',
                     TfidfVectorizer(max_df=0.95, ngram_range=(1, 2),
                                     sublinear_tf=True, token_pattern=None,
                                     tokenizer=<function tokenize at 0x11c13f060>)),
                    ('clf',
                     LogisticRegression(C=3.0774851336928744,
                                        class_weight='balanced', max_iter=2000,
                                        penalty='l2', random_state=7524,
                                        solver='newton-cg',
                                        tol=0.00021806665627858358))]),
    'LinearSVC': Pipeline(steps=[('tfidf',
                     TfidfVectorizer(max_df=0.95, ngram_range=(1, 2),
                                     sublinear_tf=True, token_pattern=None,
                                     tokenizer=<function tokenize at 0x11c13f060>)),
                    ('clf',
                     Li

# Simple Ensemble

In [12]:
gaming_model_collection = GamingModelCollection(
    model_joblibs = model_paths,
)

In [13]:
gaming_model_collection.classifiers.keys()

dict_keys(['wot_Logistic_Regression', 'wot_LinearSVC', 'wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=False)', 'wot_LinearSVC_+_Calibrated(isotonic,_ensemble=False)', 'dota_Logistic_Regression', 'dota_LinearSVC', 'dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=True)', 'dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True)', 'combined_Logistic_Regression', 'combined_LinearSVC', 'combined_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False)', 'combined_LinearSVC_+_Calibrated(isotonic,_ensemble=True)'])

In [14]:
# instantiate the ensemble with model paths
multiclass_ensemble = ensemble.Ensemble(
    model_collections=[gaming_model_collection]
)

In [15]:
# run simple majority voting on the training data
pred_train_multiclass_ensemble = multiclass_ensemble.predict_simple_majority(train[tc])

Predicting with SimpleMajority...


In [16]:
_prediction_metrics(train['label_3class'], pred_train_multiclass_ensemble)

{'CV Macro F1': 0.9160356887082272,
 'CV Weighted F1': 0.9589583920446748,
 'Accuracy': 0.9593721488818963,
 'Precision': np.float64(0.9918325428975101),
 'FPR': np.float64(0.007638462483114331),
 'FNR': np.float64(0.07240704500978473),
 'TPR': np.float64(0.9275929549902152),
 'TNR': np.float64(0.9923615375168857)}

# Fitted Ensemble

In [19]:
thresholds = np.arange(0.50, 0.99, 0.05)

In [20]:
combined_val_pred = multiclass_ensemble.fit_weighted_confidence_majority(X_val=train[tc], y_val=train['label_3class'], score_func=score_func, thresholds=thresholds)

In [21]:
combined_val_pred

(array([0.03619482, 0.06702476, 0.00164568, 0.08790234, 0.10036961,
        0.01326474, 0.02300475, 0.02359207, 0.35831853, 0.01897598,
        0.24716295, 0.02254375]),
 0.6935912212727603)

In [22]:
_prediction_metrics(train['label_3class'], pred_train_multiclass_ensemble)

{'CV Macro F1': 0.9160356887082272,
 'CV Weighted F1': 0.9589583920446748,
 'Accuracy': 0.9593721488818963,
 'Precision': np.float64(0.9918325428975101),
 'FPR': np.float64(0.007638462483114331),
 'FNR': np.float64(0.07240704500978473),
 'TPR': np.float64(0.9275929549902152),
 'TNR': np.float64(0.9923615375168857)}

# Hold-Out Performance

In [23]:
combined_pred = multiclass_ensemble.predict_weighted_confidence_majority(X=val[tc])

In [24]:
_prediction_metrics(val['label_3class'], combined_pred)

{'CV Macro F1': 0.7126585181425368,
 'CV Weighted F1': 0.8630460569670835,
 'Accuracy': 0.8663813320825516,
 'Precision': np.float64(0.9465774151139696),
 'FPR': np.float64(0.0382126817386009),
 'FNR': np.float64(0.32292191435768264),
 'TPR': np.float64(0.6770780856423174),
 'TNR': np.float64(0.9617873182613991)}

# View Hist

In [27]:
res = multiclass_ensemble.weighted_confidence_majority.weight_history

# for each weight and threshold combination, get the _prediction metrics on the validation set
metrics_history = []

for entry in res:
    weights = entry['weights']
    threshold = entry['threshold']
    pred = multiclass_ensemble.predict_weighted_confidence_majority(X=val[tc], weights=weights, threshold=threshold)
    metrics = _prediction_metrics(val['label_3class'], pred)
    metrics_history.append({
        'weights': weights,
        'threshold': threshold,
        **metrics
    })
    
metrics_df = pd.DataFrame(metrics_history)

KeyboardInterrupt: 

In [ ]:
metrics_df.sort_values('CV Macro F1', ascending=False).head(10)